# Hyperparameter tuning

# GridSearch cv

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

# Load dataset
df = pd.read_csv(r"C:\Users\sp719\OneDrive\Desktop\project\ipl.csv", low_memory=False)

# Remove rows where winner is missing
df = df.dropna(subset=['winner'])

# Features
features = [
    'runs_scored',
    'extras',
    'current_score',
    'wickets_down',
    'balls_remaining',
    'wickets_remaining',
    'current_run_rate',
    'required_run_rate',
    'target_score',
    'batting_team',
    'bowling_team'
]

# Convert team names into numbers
le1 = LabelEncoder()
le2 = LabelEncoder()

df['batting_team'] = le1.fit_transform(df['batting_team'].astype(str))
df['bowling_team'] = le2.fit_transform(df['bowling_team'].astype(str))

# Input and output
X = df[features]

# Replace infinite/missing values
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(0)

y = df['winner']

# Take a smaller sample for faster execution
X_sample, _, y_sample, _ = train_test_split(
    X, y, train_size=10000,
    stratify=y,
    random_state=42
)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_sample, y_sample,
    test_size=0.2,
    stratify=y_sample,
    random_state=42
)

# Random Forest model
model = RandomForestClassifier(random_state=42)

# Hyperparameters to test
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, None]
}

# GridSearchCV
grid = GridSearchCV(
    model,
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=1
)

# Perform tuning
grid.fit(X_train, y_train)

# Best parameters
print("Best Parameters:", grid.best_params_)

# Best cross-validation score
print("Best CV Accuracy:", grid.best_score_)

# Test accuracy
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))

Best Parameters: {'max_depth': None, 'n_estimators': 100}
Best CV Accuracy: 0.7355006168114957
Test Accuracy: 0.7745
